# Group 13 ANN Experiments

This notebook prepares and verifies the Google Colab environment for the Group 13 project. It uses **SRSD-Feynman Medium only**. It mounts Drive, syncs the team repository, verifies dependencies and dataset files, then runs the existing test suite.

Run the sections in order. Set the Colab runtime to GPU before starting if you plan to train later. Pilot and full experiment cells are intentionally deferred until the shared model, data, metrics, runner, and aggregation modules are implemented and integrated.

## 1. Environment setup and hardware check

In [ ]:
import os, platform, random, subprocess, sys
from pathlib import Path

try:
    import google.colab  # noqa: F401
except ImportError as exc:
    raise RuntimeError('Run this notebook in Google Colab.') from exc

print('Python:', platform.python_version())
try:
    import torch
except ImportError as exc:
    raise RuntimeError('PyTorch is missing. Select a standard Colab runtime; this notebook avoids reinstalling the large Colab PyTorch build.') from exc

print('PyTorch:', torch.__version__)
print('CUDA build:', torch.version.cuda)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    subprocess.run(['nvidia-smi'], check=False)
else:
    print('GPU: none (CPU execution is available, but training will be slower)')

SEED = 13
random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = False
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device selected:', DEVICE, '| Seed:', SEED)

## 2. Google Drive and persistent project folders

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT = Path('/content/drive/MyDrive/ann-group13')
DATA_ROOT = DRIVE_ROOT / 'data' / 'srsd-feynman_medium'
RESULTS_DIR = DRIVE_ROOT / 'results'
CHECKPOINTS_DIR = DRIVE_ROOT / 'checkpoints'
FIGURES_DIR = DRIVE_ROOT / 'figures'
for folder in (DATA_ROOT.parent, RESULTS_DIR, CHECKPOINTS_DIR, FIGURES_DIR):
    folder.mkdir(parents=True, exist_ok=True)
os.environ['SRSD_ROOT'] = str(DATA_ROOT)
print('Drive project directory:', DRIVE_ROOT)
print('SRSD_ROOT:', os.environ['SRSD_ROOT'])
print('Persistent outputs:', RESULTS_DIR, CHECKPOINTS_DIR, FIGURES_DIR)

## 3. Repository access and dependency verification
The repository is public, so no token or credential is needed. Change `REPO_BRANCH` only when the team identifies its active development branch. The default below is the inspected repository default, `master`.

In [ ]:
import importlib.util, importlib.metadata, shutil

REPO_URL = 'https://github.com/Elbehery-237/ann-group13.git'
REPO_BRANCH = 'phase-c-verification'
REPO = Path('/content/ann-group13')  # code is re-created from GitHub each Colab runtime
if REPO.exists() and (REPO / '.git').exists():
    subprocess.run(['git', '-C', str(REPO), 'fetch', 'origin', REPO_BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO), 'checkout', REPO_BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only', 'origin', REPO_BRANCH], check=True)
elif REPO.exists():
    raise RuntimeError(f'{REPO} exists but is not a Git checkout; remove or rename it and rerun this section.')
else:
    subprocess.run(['git', 'clone', '--branch', REPO_BRANCH, '--single-branch', REPO_URL, str(REPO)], check=True)
os.chdir(REPO)
print('Repository:', REPO, '| branch:', REPO_BRANCH)

# Install the repository's small additional dependencies; keep Colab's torch/numpy stack.
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(REPO / 'requirements.txt')], check=True)
if importlib.util.find_spec('pytest') is None:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'pytest'], check=True)

import numpy, sympy, pandas, matplotlib, pytest
import zss, kan
print('NumPy:', numpy.__version__)
print('SymPy:', sympy.__version__)
print('pandas:', pandas.__version__)
print('matplotlib:', matplotlib.__version__)
print('pytest:', pytest.__version__)
print('zss:', importlib.metadata.version('zss'))
print('pykan:', importlib.metadata.version('pykan'))

## 4. Download and validate SRSD-Feynman Medium
The dataset is stored on Drive so it survives a runtime reset. This cell rejects a different dataset, checks all 40 equation IDs and split dimensions, opens each ground-truth pickle, and probes target standardization using training statistics in float64. The scaling probe checks numerical safety; the eventual production loader remains the shared `src/data.py` contract.

In [ ]:
import pickle
import numpy as np

DATASET_URL = 'https://huggingface.co/datasets/yoshitomo-matsubara/srsd-feynman_medium'
if not shutil.which('git-lfs'):
    subprocess.run(['apt-get', 'update', '-qq'], check=True)
    subprocess.run(['apt-get', 'install', '-y', '-qq', 'git-lfs'], check=True)
subprocess.run(['git', 'lfs', 'install'], check=True)
if not DATA_ROOT.exists():
    subprocess.run(['git', 'clone', DATASET_URL, str(DATA_ROOT)], check=True)
elif not (DATA_ROOT / '.git').exists():
    raise RuntimeError(f'{DATA_ROOT} exists but is not the expected dataset checkout. Inspect it before replacing it.')
else:
    subprocess.run(['git', '-C', str(DATA_ROOT), 'lfs', 'pull'], check=True)

split_dirs = {split: DATA_ROOT / split for split in ('train', 'val', 'test')}
eq_dir = DATA_ROOT / 'true_eq'
assert all(path.is_dir() for path in split_dirs.values()), 'Missing train/val/test directory'
assert eq_dir.is_dir(), 'Missing true_eq directory'
equations = {p.stem for p in eq_dir.glob('*.pkl')}
assert len(equations) == 40, f'Expected 40 Medium equations; found {len(equations)}'
for split, expected_rows in (('train', 8000), ('val', 1000), ('test', 1000)):
    files = {p.stem: p for p in split_dirs[split].glob('*.txt')}
    assert set(files) == equations, f'{split} equation IDs differ from true_eq'
    for name, path in files.items():
        arr = np.loadtxt(path, dtype=np.float64)
        assert arr.ndim == 2 and arr.shape[0] == expected_rows, f'{path.name}: unexpected shape {arr.shape}'
        d = arr.shape[1] - 1
        assert 1 <= d <= 6, f'{path.name}: expected 1–6 input columns, got {d}'
        assert np.isfinite(arr).all(), f'{path.name}: non-finite values found'
    print(f'{split}: {len(files)} files, {expected_rows} rows per file')

# Check all ground-truth pickles are readable and target standardization remains finite.
for name in sorted(equations):
    with (eq_dir / f'{name}.pkl').open('rb') as f:
        _ = pickle.load(f)
    y_train = np.loadtxt(split_dirs['train'] / f'{name}.txt', dtype=np.float64)[:, -1]
    mean, std = y_train.mean(dtype=np.float64), y_train.std(dtype=np.float64)
    assert np.isfinite(mean) and np.isfinite(std) and std > 0, f'{name}: invalid train target statistics'
    for split in ('train', 'val', 'test'):
        y = np.loadtxt(split_dirs[split] / f'{name}.txt', dtype=np.float64)[:, -1]
        y_scaled = ((y - mean) / std).astype(np.float32)
        assert np.isfinite(y_scaled).all(), f'{name}/{split}: target scaling overflowed float32'

os.environ['SRSD_ROOT'] = str(DATA_ROOT)
print('Validated Medium dataset:', len(equations), 'equations at', DATA_ROOT)

## 5. Automated tests
Run from the repository root and show pytest skip reasons. A non-zero exit or any skip must be reported as such; do not start experiments after a failure until it is understood.

In [ ]:
os.chdir(REPO)
test_run = subprocess.run([sys.executable, '-m', 'pytest', '-q', '-rs'], text=True)
print('pytest exit code:', test_run.returncode)
if test_run.returncode != 0:
    raise RuntimeError('Existing test suite failed; inspect the pytest output above.')

## 6. Pilot experiments — deferred

Do not launch a pilot yet. The repository does not currently contain an implemented CKAN model, registered model constructors, a production data loader, metrics, or an experiment runner. Once those shared interfaces exist and their tests pass, add a small pilot here for CK-2 and CK-3, multiple seeds, CSV persistence, and checkpoint recovery.

## 7. Full experiments — deferred

Run the full Medium benchmark only after the pilot is reviewed and the official Yousef Plan confirms the final specifications. Keep all checkpoints and results under `DRIVE_ROOT`.

## 8. Results and figures — deferred

Create summaries and figures only from completed experiment outputs. No results or figures are generated by this setup notebook.